# Database Creation

This notebook creates a spatial PostgreSQL/PostGIS database for storing and querying the cleaned OpenStreetMap building data.

**Input**
- Cleaned OpenStreetMap building data for Germany.

**Output**
- A PostGIS building database for spatial storage and querying.

In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine
import getpass

In [ ]:

# Get current cluster username
username = getpass.getuser()


# Connect to the database on our custom port
engine = create_engine(f"postgresql://{username}@localhost:5433/osm_buildings")

In [ ]:

print("Loading Parquet file...")
gdf_bldg = gpd.read_parquet(os.path.join(DATA_DIR, 'interim', 'germany_buildings_cleaned02.parquet'))

print("Pushing data to PostGIS...")
# Push to database
gdf_bldg.to_postgis(
    name='germany_buildings', 
    con=engine, 
    if_exists='replace',      
    index=False
)

print("Data successfully loaded into PostGIS!")

Loading Parquet file...
Pushing data to PostGIS...
Data successfully loaded into PostGIS!


In [ ]:
import pandas as pd

# Ask the database to count the rows
count = pd.read_sql("SELECT count(*) FROM germany_buildings;", con=engine)
print(f"Total buildings in database: {count.iloc[0,0]}")

# Look at the first 3 rows just to see what they look like
preview = pd.read_sql("SELECT * FROM germany_buildings LIMIT 3;", con=engine)
display(preview)

Total buildings in database: 38802424


,addr:city,addr:housenumber,addr:postcode,addr:street,name,building,amenity,building:use,craft,office,shop,id,tags,geometry,building:levels,address_tier,area_sqm,perimeter_m,compactness
0,Weinheim,5,69469,Babostraße,Konferenzhaus Unternehmensgruppe Freudenberg,yes,None,None,None,None,None,3428357,None,0103000020E6100000010000000C000000000000601257...,None,2,370.210483,82.562067,0.682492
1,None,None,None,None,Reithalle,riding_hall,None,None,None,None,None,3453425,"{""roof:shape"":""gabled"",""sport"":""equestrian""}",0103000020E6100000010000000500000000000020C1CD...,None,0,982.378210,131.550152,0.713355
2,None,None,None,None,None,yes,None,None,None,None,None,3453428,"{""created_by"":""JOSM""}",0103000020E61000000100000005000000000000201ECD...,None,0,85.275713,37.287339,0.770748


In [ ]:

# Write a pure SQL query to group and count the data
sql_query = """
SELECT building, COUNT(*) as total_count
FROM germany_buildings
GROUP BY building
ORDER BY total_count DESC
LIMIT 10;
"""

# Ask the database to do the heavy lifting and just return the tiny result table
top_buildings = pd.read_sql(sql_query, con=engine)
display(top_buildings)

,building,total_count
0,yes,25834156
1,house,3234432
2,garage,2525391
3,apartments,1465511
4,residential,1163093
5,detached,1087764
6,shed,455610
7,semidetached_house,418574
8,garages,391009
9,roof,297787
